# NB_bishop_ch20_diffusion_models

**Chapter 20 — Diffusion Models**

This notebook implements the forward diffusion process, visualizes the noising schedule, trains a simple DDPM on MNIST, shows the denoising process, and introduces the classifier-free guidance concept.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_20/NB_bishop_ch20_diffusion_models.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## 1. Forward Diffusion Process

The forward process progressively adds Gaussian noise to data over $T$ timesteps:

$$q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t; \sqrt{\bar{\alpha}_t}\, \mathbf{x}_0,\; (1 - \bar{\alpha}_t)\, \mathbf{I})$$

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
# Scale to [-1, 1]
X_train = X_train * 2.0 - 1.0
X_test = X_test * 2.0 - 1.0
X_train = X_train[..., np.newaxis]  # (N, 28, 28, 1)
X_test = X_test[..., np.newaxis]

print(f'Training data: {X_train.shape}')
print(f'Value range: [{X_train.min():.1f}, {X_train.max():.1f}]')

In [ ]:
# Diffusion schedule parameters
T = 1000  # total timesteps
beta_start = 1e-4
beta_end = 0.02

# Linear beta schedule
betas = np.linspace(beta_start, beta_end, T).astype(np.float32)
alphas = 1.0 - betas
alphas_cumprod = np.cumprod(alphas)  # alpha_bar_t
sqrt_alphas_cumprod = np.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - alphas_cumprod)

print(f'beta range: [{betas[0]:.5f}, {betas[-1]:.5f}]')
print(f'alpha_bar at t=0: {alphas_cumprod[0]:.5f}')
print(f'alpha_bar at t=T: {alphas_cumprod[-1]:.5f}')

In [ ]:
def forward_diffusion(x0, t, noise=None):
    """Add noise to x0 at timestep t using the closed-form formula."""
    if noise is None:
        noise = np.random.randn(*x0.shape).astype(np.float32)
    sqrt_alpha_bar = sqrt_alphas_cumprod[t]
    sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t]
    return sqrt_alpha_bar * x0 + sqrt_one_minus * noise, noise

In [ ]:
# Visualize forward diffusion at various timesteps
sample_img = X_train[0:1]  # (1, 28, 28, 1)
timesteps = [0, 50, 100, 200, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps), figsize=(16, 2.5))
for i, t in enumerate(timesteps):
    noisy, _ = forward_diffusion(sample_img, t)
    axes[i].imshow(np.clip((noisy[0, :, :, 0] + 1) / 2, 0, 1), cmap='gray')
    axes[i].set_title(f't = {t}')
    axes[i].axis('off')

fig.suptitle('Forward Diffusion: Progressive Noising', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch20_forward_diffusion')
plt.show()

## 2. Noise Schedule Visualization

In [ ]:
# Compare linear, cosine, and quadratic noise schedules
# Linear
betas_linear = np.linspace(beta_start, beta_end, T)
alpha_bar_linear = np.cumprod(1 - betas_linear)

# Cosine schedule (Nichol & Dhariwal, 2021)
s = 0.008
steps = np.arange(T + 1) / T
f_t = np.cos((steps + s) / (1 + s) * np.pi / 2) ** 2
alpha_bar_cosine = f_t / f_t[0]
betas_cosine = np.clip(1 - alpha_bar_cosine[1:] / alpha_bar_cosine[:-1], 0, 0.999)
alpha_bar_cosine = alpha_bar_cosine[1:]

# Quadratic
betas_quad = np.linspace(beta_start**0.5, beta_end**0.5, T) ** 2
alpha_bar_quad = np.cumprod(1 - betas_quad)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(T), betas_linear, label='Linear', color='#1a3a6e', linewidth=1.5)
axes[0].plot(range(T), betas_cosine, label='Cosine', color='#cd0000', linewidth=1.5)
axes[0].plot(range(T), betas_quad, label='Quadratic', color='#228B22', linewidth=1.5)
axes[0].set_xlabel('Timestep $t$')
axes[0].set_ylabel('$\\beta_t$')
axes[0].set_title('Noise Schedule: $\\beta_t$')
axes[0].legend()

axes[1].plot(range(T), alpha_bar_linear, label='Linear', color='#1a3a6e', linewidth=1.5)
axes[1].plot(range(T), alpha_bar_cosine, label='Cosine', color='#cd0000', linewidth=1.5)
axes[1].plot(range(T), alpha_bar_quad, label='Quadratic', color='#228B22', linewidth=1.5)
axes[1].set_xlabel('Timestep $t$')
axes[1].set_ylabel('$\\bar{\\alpha}_t$')
axes[1].set_title('Cumulative Signal Retention: $\\bar{\\alpha}_t$')
axes[1].legend()

fig.suptitle('Noise Schedules for Diffusion Models', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch20_noise_schedule')
plt.show()

In [ ]:
# SNR (Signal-to-Noise Ratio) for each schedule
snr_linear = alpha_bar_linear / (1 - alpha_bar_linear + 1e-10)
snr_cosine = alpha_bar_cosine / (1 - alpha_bar_cosine + 1e-10)
snr_quad = alpha_bar_quad / (1 - alpha_bar_quad + 1e-10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(range(T), snr_linear, label='Linear', color='#1a3a6e', linewidth=1.5)
ax.semilogy(range(T), snr_cosine, label='Cosine', color='#cd0000', linewidth=1.5)
ax.semilogy(range(T), snr_quad, label='Quadratic', color='#228B22', linewidth=1.5)
ax.set_xlabel('Timestep $t$')
ax.set_ylabel('SNR (log scale)')
ax.set_title('Signal-to-Noise Ratio vs Timestep')
ax.legend()
fig.tight_layout()
plt.show()

## 3. U-Net Denoising Network

In [ ]:
# Sinusoidal timestep embedding
def sinusoidal_embedding(t, dim=32):
    """Create sinusoidal position embeddings for timesteps."""
    half_dim = dim // 2
    emb = np.log(10000) / (half_dim - 1)
    emb = tf.exp(tf.range(half_dim, dtype=tf.float32) * -emb)
    emb = tf.cast(t, tf.float32)[:, None] * emb[None, :]
    emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=-1)
    return emb

print('Sinusoidal embedding defined.')

In [ ]:
def build_unet(img_shape=(28, 28, 1), time_emb_dim=32):
    """Simple U-Net for MNIST denoising."""
    # Inputs
    x_input = layers.Input(shape=img_shape, name='image')
    t_input = layers.Input(shape=(), dtype=tf.int32, name='timestep')
    
    # Time embedding
    t_emb = layers.Lambda(lambda t: sinusoidal_embedding(t, time_emb_dim))(t_input)
    t_emb = layers.Dense(64, activation='relu')(t_emb)
    t_emb = layers.Dense(64, activation='relu')(t_emb)
    
    # Encoder
    # Block 1: 28x28 -> 14x14
    h1 = layers.Conv2D(32, 3, padding='same', activation='relu')(x_input)
    h1 = layers.Conv2D(32, 3, padding='same', activation='relu')(h1)
    # Add time embedding
    t1 = layers.Dense(32)(t_emb)
    t1 = layers.Reshape((1, 1, 32))(t1)
    h1 = h1 + t1
    p1 = layers.MaxPooling2D(2)(h1)
    
    # Block 2: 14x14 -> 7x7
    h2 = layers.Conv2D(64, 3, padding='same', activation='relu')(p1)
    h2 = layers.Conv2D(64, 3, padding='same', activation='relu')(h2)
    t2 = layers.Dense(64)(t_emb)
    t2 = layers.Reshape((1, 1, 64))(t2)
    h2 = h2 + t2
    p2 = layers.MaxPooling2D(2)(h2)
    
    # Bottleneck: 7x7
    bn = layers.Conv2D(128, 3, padding='same', activation='relu')(p2)
    bn = layers.Conv2D(128, 3, padding='same', activation='relu')(bn)
    t_bn = layers.Dense(128)(t_emb)
    t_bn = layers.Reshape((1, 1, 128))(t_bn)
    bn = bn + t_bn
    
    # Decoder
    # Block 3: 7x7 -> 14x14
    u2 = layers.UpSampling2D(2)(bn)
    u2 = layers.Concatenate()([u2, h2])
    u2 = layers.Conv2D(64, 3, padding='same', activation='relu')(u2)
    u2 = layers.Conv2D(64, 3, padding='same', activation='relu')(u2)
    
    # Block 4: 14x14 -> 28x28
    u1 = layers.UpSampling2D(2)(u2)
    u1 = layers.Concatenate()([u1, h1])
    u1 = layers.Conv2D(32, 3, padding='same', activation='relu')(u1)
    u1 = layers.Conv2D(32, 3, padding='same', activation='relu')(u1)
    
    # Output: predict noise
    output = layers.Conv2D(1, 1, padding='same')(u1)
    
    model = keras.Model([x_input, t_input], output, name='unet')
    return model

unet = build_unet()
unet.summary()

## 4. Training the DDPM

In [ ]:
# Convert schedule to TF tensors
betas_tf = tf.constant(betas)
alphas_tf = tf.constant(alphas)
alphas_cumprod_tf = tf.constant(alphas_cumprod)
sqrt_alphas_cumprod_tf = tf.constant(sqrt_alphas_cumprod)
sqrt_one_minus_tf = tf.constant(sqrt_one_minus_alphas_cumprod)

optimizer = keras.optimizers.Adam(learning_rate=2e-4)

BATCH_SIZE = 128
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(BATCH_SIZE)

In [ ]:
@tf.function
def train_step(x0):
    batch_size = tf.shape(x0)[0]
    # Sample random timesteps
    t = tf.random.uniform([batch_size], 0, T, dtype=tf.int32)
    
    # Sample noise
    noise = tf.random.normal(tf.shape(x0))
    
    # Create noisy images
    sqrt_ab = tf.gather(sqrt_alphas_cumprod_tf, t)
    sqrt_ab = tf.reshape(sqrt_ab, [-1, 1, 1, 1])
    sqrt_om = tf.gather(sqrt_one_minus_tf, t)
    sqrt_om = tf.reshape(sqrt_om, [-1, 1, 1, 1])
    
    x_noisy = sqrt_ab * x0 + sqrt_om * noise
    
    with tf.GradientTape() as tape:
        noise_pred = unet([x_noisy, t], training=True)
        loss = tf.reduce_mean(tf.square(noise - noise_pred))
    
    grads = tape.gradient(loss, unet.trainable_variables)
    optimizer.apply_gradients(zip(grads, unet.trainable_variables))
    return loss

In [ ]:
# Train DDPM
EPOCHS = 20
losses = []

for epoch in range(EPOCHS):
    start = time.time()
    epoch_losses = []
    for batch in dataset:
        loss = train_step(batch)
        epoch_losses.append(loss.numpy())
    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.5f} — {time.time()-start:.1f}s')

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, EPOCHS+1), losses, 'o-', color='#1a3a6e', linewidth=1.5, markersize=5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('DDPM Training Loss')
fig.tight_layout()
plt.show()

## 5. Sampling (Reverse Diffusion)

In [ ]:
def ddpm_sample(model, n_samples=16, img_shape=(28, 28, 1)):
    """Generate samples via reverse diffusion (DDPM sampling)."""
    # Start from pure noise
    x = tf.random.normal([n_samples] + list(img_shape))
    
    intermediates = []
    save_at = [999, 900, 700, 500, 300, 100, 50, 10, 0]
    
    for t in reversed(range(T)):
        t_batch = tf.fill([n_samples], t)
        
        # Predict noise
        noise_pred = model([x, t_batch], training=False)
        
        # DDPM update
        alpha_t = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t = betas[t]
        
        # Compute x_{t-1}
        coeff1 = 1.0 / np.sqrt(alpha_t)
        coeff2 = beta_t / np.sqrt(1.0 - alpha_bar_t)
        mean = coeff1 * (x - coeff2 * noise_pred)
        
        if t > 0:
            noise = tf.random.normal(tf.shape(x))
            sigma = np.sqrt(beta_t)
            x = mean + sigma * noise
        else:
            x = mean
        
        if t in save_at:
            intermediates.append((t, x.numpy().copy()))
    
    return x.numpy(), intermediates

print('DDPM sampling function defined.')

In [ ]:
# Generate samples
print('Generating samples (this may take a minute)...')
generated, intermediates = ddpm_sample(unet, n_samples=64)

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    img = np.clip((generated[i, :, :, 0] + 1) / 2, 0, 1)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
fig.suptitle('DDPM — 64 Generated Digits', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch20_generated')
plt.show()

## 6. Visualize the Denoising Process

In [ ]:
# Show the denoising process for a few samples
n_show = 4
fig, axes = plt.subplots(n_show, len(intermediates), figsize=(18, 2.5 * n_show))

for col, (t, imgs) in enumerate(intermediates):
    for row in range(n_show):
        img = np.clip((imgs[row, :, :, 0] + 1) / 2, 0, 1)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
    axes[0, col].set_title(f't = {t}')

fig.suptitle('Reverse Diffusion: Denoising Process', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch20_denoising')
plt.show()

In [ ]:
# Visualize what the network predicts at different noise levels
sample = X_train[42:43]  # single image
t_values = [0, 50, 100, 200, 500, 800]

fig, axes = plt.subplots(3, len(t_values), figsize=(16, 6))
for i, t in enumerate(t_values):
    noise = np.random.randn(*sample.shape).astype(np.float32)
    noisy = sqrt_alphas_cumprod[t] * sample + sqrt_one_minus_alphas_cumprod[t] * noise
    
    t_batch = tf.constant([t], dtype=tf.int32)
    pred_noise = unet([noisy, t_batch], training=False).numpy()
    
    # Reconstruct x0 estimate
    x0_hat = (noisy - sqrt_one_minus_alphas_cumprod[t] * pred_noise) / sqrt_alphas_cumprod[t]
    
    axes[0, i].imshow(np.clip((noisy[0, :, :, 0] + 1) / 2, 0, 1), cmap='gray')
    axes[0, i].set_title(f't={t}'); axes[0, i].axis('off')
    axes[1, i].imshow(pred_noise[0, :, :, 0], cmap='RdBu_r')
    axes[1, i].axis('off')
    axes[2, i].imshow(np.clip((x0_hat[0, :, :, 0] + 1) / 2, 0, 1), cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Noisy', fontsize=12, rotation=0, labelpad=40)
axes[1, 0].set_ylabel('Pred Noise', fontsize=12, rotation=0, labelpad=50)
axes[2, 0].set_ylabel('$\\hat{x}_0$', fontsize=12, rotation=0, labelpad=30)
fig.suptitle('Network Predictions at Different Noise Levels', fontsize=14)
fig.tight_layout()
plt.show()

## 7. Classifier-Free Guidance (Concept)

Classifier-free guidance combines conditional and unconditional noise predictions:

$$\hat{\epsilon}_\theta(\mathbf{x}_t, c) = (1 + w)\, \epsilon_\theta(\mathbf{x}_t, c) - w\, \epsilon_\theta(\mathbf{x}_t, \varnothing)$$

where $w$ is the guidance scale and $c$ is the class label.

In [ ]:
def build_conditional_unet(img_shape=(28, 28, 1), n_classes=10, time_emb_dim=32):
    """U-Net with class conditioning for classifier-free guidance."""
    x_input = layers.Input(shape=img_shape, name='image')
    t_input = layers.Input(shape=(), dtype=tf.int32, name='timestep')
    c_input = layers.Input(shape=(), dtype=tf.int32, name='class_label')
    
    # Time embedding
    t_emb = layers.Lambda(lambda t: sinusoidal_embedding(t, time_emb_dim))(t_input)
    t_emb = layers.Dense(64, activation='relu')(t_emb)
    
    # Class embedding (n_classes + 1 for null class)
    c_emb = layers.Embedding(n_classes + 1, 64)(c_input)
    
    # Combined conditioning
    cond = layers.Add()([t_emb, c_emb])
    cond = layers.Dense(64, activation='relu')(cond)
    
    # Simplified U-Net with conditioning
    h = layers.Conv2D(32, 3, padding='same', activation='relu')(x_input)
    c1 = layers.Dense(32)(cond)
    c1 = layers.Reshape((1, 1, 32))(c1)
    h = h + c1
    h = layers.Conv2D(32, 3, padding='same', activation='relu')(h)
    skip1 = h
    h = layers.MaxPooling2D(2)(h)
    
    h = layers.Conv2D(64, 3, padding='same', activation='relu')(h)
    c2 = layers.Dense(64)(cond)
    c2 = layers.Reshape((1, 1, 64))(c2)
    h = h + c2
    h = layers.Conv2D(64, 3, padding='same', activation='relu')(h)
    skip2 = h
    h = layers.MaxPooling2D(2)(h)
    
    h = layers.Conv2D(128, 3, padding='same', activation='relu')(h)
    h = layers.Conv2D(128, 3, padding='same', activation='relu')(h)
    
    h = layers.UpSampling2D(2)(h)
    h = layers.Concatenate()([h, skip2])
    h = layers.Conv2D(64, 3, padding='same', activation='relu')(h)
    
    h = layers.UpSampling2D(2)(h)
    h = layers.Concatenate()([h, skip1])
    h = layers.Conv2D(32, 3, padding='same', activation='relu')(h)
    
    output = layers.Conv2D(1, 1, padding='same')(h)
    
    return keras.Model([x_input, t_input, c_input], output, name='cond_unet')

cond_unet = build_conditional_unet()
print(f'Conditional U-Net parameters: {cond_unet.count_params():,}')

In [ ]:
# Demonstrate the concept: show how guidance scale affects generation
# (Using the unconditional model as a proxy for illustration)

# The idea: with guidance scale w,
# eps_guided = (1 + w) * eps_conditional - w * eps_unconditional

print('Classifier-Free Guidance concept:')
print('='*60)
print('During training:')
print('  - Randomly drop class label (replace with null token) with prob p=0.1')
print('  - Network learns both conditional and unconditional denoising')
print()
print('During sampling with guidance scale w:')
print('  eps_guided = (1+w) * eps(x_t, c) - w * eps(x_t, null)')
print()
print('  w = 0: unconditional generation')
print('  w = 1: standard conditional generation')
print('  w > 1: amplified conditioning (higher quality, less diversity)')
print()
print('Typical values: w = 2.0 to 7.5 (e.g., Stable Diffusion uses w=7.5)')

In [ ]:
# Visualize the effect of guidance conceptually using noise scaling
# Simulate what different guidance scales would do to noise prediction
sample_img = X_train[0:1]
t_demo = 200
noise = np.random.randn(*sample_img.shape).astype(np.float32)
noisy = sqrt_alphas_cumprod[t_demo] * sample_img + sqrt_one_minus_alphas_cumprod[t_demo] * noise

t_batch = tf.constant([t_demo], dtype=tf.int32)
pred_noise = unet([noisy, t_batch], training=False).numpy()

guidance_scales = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
fig, axes = plt.subplots(1, len(guidance_scales), figsize=(16, 3))

# Simulate: use pred_noise as "conditional" and a weaker signal as "unconditional"
uncond_noise = pred_noise * 0.5 + noise * 0.5  # simulate weaker unconditional pred

for i, w in enumerate(guidance_scales):
    guided = (1 + w) * pred_noise - w * uncond_noise
    x0_hat = (noisy - sqrt_one_minus_alphas_cumprod[t_demo] * guided) / sqrt_alphas_cumprod[t_demo]
    img = np.clip((x0_hat[0, :, :, 0] + 1) / 2, 0, 1)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'w = {w}')
    axes[i].axis('off')

fig.suptitle('Effect of Guidance Scale on Reconstruction', fontsize=14)
fig.tight_layout()
plt.show()

## Summary

In this notebook we covered:
- **Forward diffusion** — progressively adding Gaussian noise to images
- **Noise schedules** — linear, cosine, and quadratic schedules with SNR analysis
- **U-Net architecture** — with sinusoidal timestep embeddings and skip connections
- **DDPM training and sampling** — predicting and removing noise iteratively
- **Classifier-free guidance** — controlling generation quality via guidance scale

In [ ]:
# Key takeaways
takeaways = [
    '1. Diffusion models learn to reverse a gradual noising process.',
    '2. The noise schedule controls the tradeoff between training stability and sample quality.',
    '3. The U-Net predicts the noise added at each timestep, conditioned on t.',
    '4. DDPM sampling requires iterating through all T timesteps (slow but high quality).',
    '5. Classifier-free guidance trades diversity for quality by amplifying conditional signal.'
]
print('Key Takeaways:')
print('-' * 60)
for t in takeaways:
    print(t)